In [1]:
# Cell 1: Install dependencies

%pip install -q requests pandas

Note: you may need to restart the kernel to use updated packages.


In [1]:
# Cell 2: Imports and configuration

from __future__ import annotations

import json
import re
import textwrap
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import requests
from IPython.display import display

# ----------------------------
# Notebook configuration
# ----------------------------

OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "mistral:7b"   # Change this later if you want to try a different local Ollama model.
REQUEST_TIMEOUT_SECONDS = 120

# Context window: how many neighboring segments to include on each side.
CONTEXT_BACK = 1
CONTEXT_FORWARD = 1

# Deterministic / low-randomness settings for classification.
TEMPERATURE = 0.0

# Smoothing config
ENABLE_SMOOTHING = True
LOW_CONFIDENCE_THRESHOLD = 0.60

# JSON schema that we ask Ollama to follow.
CLASSIFIER_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "properties": {
        "predicted_role": {
            "type": "string",
            "enum": ["doctor", "patient"]
        },
        "confidence": {
            "type": "number"
        },
        "short_reason": {
            "type": "string"
        }
    },
    "required": ["predicted_role", "confidence", "short_reason"]
}

print("Configuration loaded.")
print(f"OLLAMA_BASE_URL = {OLLAMA_BASE_URL}")
print(f"OLLAMA_MODEL    = {OLLAMA_MODEL}")
print(f"Context window  = prev {CONTEXT_BACK}, next {CONTEXT_FORWARD}")

Configuration loaded.
OLLAMA_BASE_URL = http://localhost:11434
OLLAMA_MODEL    = mistral:7b
Context window  = prev 1, next 1


In [6]:
# Cell 3: User inputs for directory and filename

# Edit these manually before running.
project_dir = Path("transcription_outputs")
input_filename = "doctor_patient_example_segments.json"

# Output filenames will be saved in the same directory.
output_stem = "classified_segments"

# Pick one example segment index for prompt / raw-response inspection.
sample_segment_index = 0

input_path = project_dir / input_filename

print(f"Input path will be: {input_path}")

Input path will be: transcription_outputs/doctor_patient_example_segments.json


In [7]:
# Cell 4: Load and validate JSON

@dataclass
class TranscriptSegment:
    id: Any
    start: float
    end: float
    text: str


def normalize_whitespace(text: str) -> str:
    """Collapse repeated whitespace and trim edges."""
    return re.sub(r"\s+", " ", str(text)).strip()


def load_json_file(path: Path) -> Dict[str, Any]:
    """Load a JSON file from disk with basic path validation."""
    if not isinstance(path, Path):
        raise TypeError("path must be a pathlib.Path")
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    if path.suffix.lower() != ".json":
        raise ValueError(f"Expected a .json file, got: {path.name}")

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, dict):
        raise ValueError("Top-level JSON must be an object / dictionary.")

    return data


def validate_and_extract_segments(data: Dict[str, Any]) -> List[TranscriptSegment]:
    """Validate the minimum expected schema and convert segment records into typed objects."""
    if "segments" not in data:
        raise KeyError("Top-level key 'segments' was not found.")
    if not isinstance(data["segments"], list):
        raise TypeError("Top-level 'segments' must be a list.")

    cleaned_segments: List[TranscriptSegment] = []

    required_keys = {"id", "start", "end", "text"}

    for idx, raw_seg in enumerate(data["segments"]):
        if not isinstance(raw_seg, dict):
            raise TypeError(f"Segment at index {idx} is not a dictionary.")

        missing = required_keys - set(raw_seg.keys())
        if missing:
            raise KeyError(f"Segment at index {idx} is missing required keys: {sorted(missing)}")

        try:
            seg = TranscriptSegment(
                id=raw_seg["id"],
                start=float(raw_seg["start"]),
                end=float(raw_seg["end"]),
                text=normalize_whitespace(raw_seg["text"]),
            )
        except Exception as exc:
            raise ValueError(f"Failed to coerce segment at index {idx}: {exc}") from exc

        cleaned_segments.append(seg)

    if not cleaned_segments:
        raise ValueError("The JSON file contains zero segments.")

    return cleaned_segments


transcript_data = load_json_file(input_path)
segments = validate_and_extract_segments(transcript_data)

print(f"Loaded {len(segments)} segments from: {input_path}")

Loaded 21 segments from: transcription_outputs/doctor_patient_example_segments.json


In [8]:
# Cell 5: Inspect schema and preview segments

def top_level_schema_summary(data: Dict[str, Any]) -> Dict[str, str]:
    """Return a quick summary of top-level keys and their Python types."""
    return {key: type(value).__name__ for key, value in data.items()}


print("Top-level schema summary:")
print(json.dumps(top_level_schema_summary(transcript_data), indent=2))

print("\nFirst raw segment exactly as loaded from the JSON file:")
print(json.dumps(transcript_data["segments"][0], indent=2, ensure_ascii=False))

print("\nFirst few normalized segment objects:")
preview_df = pd.DataFrame([asdict(seg) for seg in segments[:5]])
display(preview_df)

Top-level schema summary:
{
  "audio_path": "str",
  "language": "str",
  "model": "str",
  "segments": "list"
}

First raw segment exactly as loaded from the JSON file:
{
  "id": 0,
  "start": 0.0,
  "end": 4.8,
  "text": " Okay, I'm the doctor."
}

First few normalized segment objects:


,id,start,end,text
0,0,0.00,4.80,"Okay, I'm the doctor."
1,1,4.80,5.84,Good morning.
2,2,5.84,7.32,Can you tell me what brings you in today?
3,3,7.32,11.84,"Uh, yeah, I've been refilling really tired lat..."
4,4,11.84,12.84,exhausted.


In [9]:
# Cell 6: Define prompt and local model call

def build_context_text(
    segments: List[TranscriptSegment],
    target_index: int,
    back: int = 1,
    forward: int = 1
) -> str:
    """
    Build the small sliding context window shown to the model.
    The target segment is labeled TARGET, and neighbors are PREV or NEXT.
    """
    start_idx = max(0, target_index - back)
    end_idx = min(len(segments), target_index + forward + 1)

    lines: List[str] = []

    for idx in range(start_idx, end_idx):
        seg = segments[idx]
        if idx < target_index:
            tag = "PREV"
        elif idx == target_index:
            tag = "TARGET"
        else:
            tag = "NEXT"

        lines.append(
            f"[{tag} id={seg.id} start={seg.start:.2f} end={seg.end:.2f}] "
            f"{normalize_whitespace(seg.text)}"
        )

    return "\n".join(lines)


def build_classifier_prompt(
    segments: List[TranscriptSegment],
    target_index: int,
    back: int = 1,
    forward: int = 1
) -> str:
    """
    Build a strict classification prompt.
    The model is told to use content cues and the local context window.
    """
    context_text = build_context_text(segments, target_index, back=back, forward=forward)

    prompt = f"""
You are a strict binary classifier for transcript segments from a medical conversation.

Task:
Determine whether the TARGET segment was spoken by the doctor or the patient.

Use the TARGET segment plus the nearby context.
Do NOT rely on speaker diarization labels.
Do NOT invent facts outside the given text.

General cues:
- Doctor speech may include questions about symptoms, medication, history, tests, assessment, advice, reassurance, and next steps.
- Patient speech may include symptoms, lived experience, feelings, habits, timing, history, and answers to intake questions.
- Very short utterances like "okay", "I see", "alright", "mm-hmm", "don't worry" must be interpreted using the surrounding context.

Return STRICT JSON ONLY.
The JSON must contain exactly these fields:
- predicted_role
- confidence
- short_reason

Rules:
- predicted_role must be exactly "doctor" or "patient"
- confidence must be a numeric value from 0 to 1
- short_reason must be one sentence or less
- no markdown
- no extra prose
- no extra keys

Context window:
{context_text}
""".strip()

    return prompt


def check_ollama_server(base_url: str = OLLAMA_BASE_URL) -> Dict[str, Any]:
    """
    Query Ollama's /api/tags endpoint to confirm the local server is reachable
    and to inspect what models are available locally.
    """
    url = f"{base_url}/api/tags"

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as exc:
        raise RuntimeError(
            "Could not reach Ollama at http://localhost:11434.\n"
            "Make sure Ollama is installed, running, and reachable from this notebook."
        ) from exc

    payload = response.json()
    models = payload.get("models", [])
    available_model_names = [m.get("name", "") for m in models if isinstance(m, dict)]

    return {
        "ok": True,
        "available_models": available_model_names,
        "raw": payload,
    }


def call_ollama_generate(
    prompt: str,
    model_name: str = OLLAMA_MODEL,
    base_url: str = OLLAMA_BASE_URL,
    timeout_seconds: int = REQUEST_TIMEOUT_SECONDS,
    schema: Optional[Dict[str, Any]] = None,
) -> Tuple[str, Dict[str, Any]]:
    """
    Call Ollama through the local HTTP API.
    We use /api/generate and ask for non-streaming output.
    """
    url = f"{base_url}/api/generate"

    payload: Dict[str, Any] = {
        "model": model_name,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": TEMPERATURE
        }
    }

    # Asking for schema-constrained JSON output improves reliability.
    if schema is not None:
        payload["format"] = schema

    try:
        response = requests.post(url, json=payload, timeout=timeout_seconds)
        response.raise_for_status()
    except requests.exceptions.RequestException as exc:
        raise RuntimeError(
            f"Ollama request failed for model '{model_name}'.\n"
            f"If the model is missing locally, try running outside the notebook:\n"
            f"ollama pull {model_name}"
        ) from exc

    response_payload = response.json()

    if "response" not in response_payload:
        raise ValueError(
            "Ollama returned a payload without a 'response' field.\n"
            f"Raw payload keys: {list(response_payload.keys())}"
        )

    return response_payload["response"], response_payload


def extract_first_json_object(text: str) -> Optional[str]:
    """
    Extract the first JSON-like object from a string.
    This helps when the model wraps JSON with extra text.
    """
    text = text.strip()

    if text.startswith("{") and text.endswith("}"):
        return text

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        return match.group(0)

    return None


def normalize_model_result(obj: Dict[str, Any]) -> Dict[str, Any]:
    """
    Validate and normalize the model's output into a strict Python dictionary.
    """
    role = str(obj.get("predicted_role", "")).strip().lower()

    if role not in {"doctor", "patient"}:
        raise ValueError(f"Invalid predicted_role: {role}")

    try:
        confidence = float(obj.get("confidence", 0.0))
    except Exception:
        confidence = 0.0

    confidence = max(0.0, min(1.0, confidence))

    short_reason = normalize_whitespace(obj.get("short_reason", ""))
    if not short_reason:
        short_reason = "No reason returned."

    return {
        "predicted_role": role,
        "confidence": confidence,
        "short_reason": short_reason[:180],
    }


def parse_model_output(raw_text: str) -> Optional[Dict[str, Any]]:
    """
    Parse the model output into the expected structured form.
    Returns None if parsing fails.
    """
    candidate = extract_first_json_object(raw_text)
    if candidate is None:
        return None

    try:
        parsed = json.loads(candidate)
        return normalize_model_result(parsed)
    except Exception:
        return None


def build_repair_prompt(bad_output: str) -> str:
    """
    If the original output is malformed, ask the same local model to repair it into valid JSON.
    """
    schema_text = json.dumps(CLASSIFIER_SCHEMA, ensure_ascii=False)

    repair_prompt = f"""
Convert the text below into valid JSON only.

Required JSON schema:
{schema_text}

Rules:
- predicted_role must be exactly "doctor" or "patient"
- confidence must be numeric and between 0 and 1
- short_reason must be a short string
- return JSON only
- no markdown
- no explanation

Text to repair:
{bad_output}
""".strip()

    return repair_prompt


def parse_with_one_repair(
    raw_text: str,
    model_name: str = OLLAMA_MODEL
) -> Tuple[Dict[str, Any], str, bool]:
    """
    Try parsing once.
    If parsing fails, retry once with a repair prompt.
    If it still fails, return a safe placeholder record.
    """
    parsed = parse_model_output(raw_text)
    if parsed is not None:
        return parsed, raw_text, False

    repair_prompt = build_repair_prompt(raw_text)
    repaired_raw_text, _ = call_ollama_generate(
        prompt=repair_prompt,
        model_name=model_name,
        schema=CLASSIFIER_SCHEMA
    )

    repaired = parse_model_output(repaired_raw_text)
    if repaired is not None:
        return repaired, repaired_raw_text, True

    fallback = {
        "predicted_role": "patient",
        "confidence": 0.01,
        "short_reason": "Fallback placeholder after parse failure."
    }
    return fallback, repaired_raw_text, True


ollama_status = check_ollama_server()
print("Ollama server is reachable.")
print("First few locally available models:")
print(ollama_status["available_models"][:10])

if OLLAMA_MODEL not in ollama_status["available_models"]:
    print(
        f"\nNote: '{OLLAMA_MODEL}' was not found in /api/tags.\n"
        f"The notebook will still try to use it, but if the next call fails, pull it with:\n"
        f"ollama pull {OLLAMA_MODEL}"
    )

Ollama server is reachable.
First few locally available models:
['mistral:7b', 'nomic-embed-text:latest', 'qwen2.5-coder:1.5b-base', 'llama3.1:8b', 'mistral:latest']


In [10]:
# Cell 7: Show exactly what is sent to the model for one example segment

# Clamp the sample index so the notebook does not fail if the chosen index is out of range.
sample_segment_index = max(0, min(sample_segment_index, len(segments) - 1))

sample_context_text = build_context_text(
    segments=segments,
    target_index=sample_segment_index,
    back=CONTEXT_BACK,
    forward=CONTEXT_FORWARD
)

sample_prompt = build_classifier_prompt(
    segments=segments,
    target_index=sample_segment_index,
    back=CONTEXT_BACK,
    forward=CONTEXT_FORWARD
)

print("EXAMPLE CONTEXT TEXT USED:\n")
print(sample_context_text)

print("\n" + "=" * 80)
print("EXACT PROMPT SENT TO OLLAMA:\n")
print(sample_prompt)

sample_raw_response, sample_http_payload = call_ollama_generate(
    prompt=sample_prompt,
    model_name=OLLAMA_MODEL,
    schema=CLASSIFIER_SCHEMA
)

print("\n" + "=" * 80)
print("RAW MODEL RESPONSE BEFORE PARSING:\n")
print(sample_raw_response)

sample_parsed, sample_final_text, sample_used_repair = parse_with_one_repair(
    raw_text=sample_raw_response,
    model_name=OLLAMA_MODEL
)

print("\n" + "=" * 80)
print("PARSED RESULT:\n")
print(json.dumps(sample_parsed, indent=2, ensure_ascii=False))
print(f"\nUsed repair prompt: {sample_used_repair}")

EXAMPLE CONTEXT TEXT USED:

[TARGET id=0 start=0.00 end=4.80] Okay, I'm the doctor.
[NEXT id=1 start=4.80 end=5.84] Good morning.

EXACT PROMPT SENT TO OLLAMA:

You are a strict binary classifier for transcript segments from a medical conversation.

Task:
Determine whether the TARGET segment was spoken by the doctor or the patient.

Use the TARGET segment plus the nearby context.
Do NOT rely on speaker diarization labels.
Do NOT invent facts outside the given text.

General cues:
- Doctor speech may include questions about symptoms, medication, history, tests, assessment, advice, reassurance, and next steps.
- Patient speech may include symptoms, lived experience, feelings, habits, timing, history, and answers to intake questions.
- Very short utterances like "okay", "I see", "alright", "mm-hmm", "don't worry" must be interpreted using the surrounding context.

Return STRICT JSON ONLY.
The JSON must contain exactly these fields:
- predicted_role
- confidence
- short_reason

Rules:
- pr

In [11]:
# Cell 8: Classify segments

def classify_one_segment(
    segments: List[TranscriptSegment],
    target_index: int,
    model_name: str = OLLAMA_MODEL,
    back: int = CONTEXT_BACK,
    forward: int = CONTEXT_FORWARD
) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """
    Classify one segment using its local context window.
    Returns:
      1. enriched output record
      2. trace info for debugging / learning
    """
    segment = segments[target_index]
    context_text = build_context_text(segments, target_index, back=back, forward=forward)
    prompt = build_classifier_prompt(segments, target_index, back=back, forward=forward)

    raw_response, raw_http_payload = call_ollama_generate(
        prompt=prompt,
        model_name=model_name,
        schema=CLASSIFIER_SCHEMA
    )

    parsed, final_text_used, used_repair = parse_with_one_repair(
        raw_text=raw_response,
        model_name=model_name
    )

    enriched_record = {
        "id": segment.id,
        "start": segment.start,
        "end": segment.end,
        "text": segment.text,
        "predicted_role": parsed["predicted_role"],
        "confidence": parsed["confidence"],
        "short_reason": parsed["short_reason"],
        "context_text_used": context_text,
    }

    trace = {
        "segment_index": target_index,
        "segment_id": segment.id,
        "prompt": prompt,
        "raw_response": raw_response,
        "final_text_used_for_parsing": final_text_used,
        "used_repair": used_repair,
    }

    return enriched_record, trace


def classify_all_segments(
    segments: List[TranscriptSegment],
    model_name: str = OLLAMA_MODEL
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    Classify every segment in sequence.
    This is intentionally simple and readable.
    """
    all_records: List[Dict[str, Any]] = []
    all_traces: List[Dict[str, Any]] = []

    total = len(segments)

    for idx in range(total):
        record, trace = classify_one_segment(
            segments=segments,
            target_index=idx,
            model_name=model_name
        )
        all_records.append(record)
        all_traces.append(trace)

        if (idx + 1) % 10 == 0 or (idx + 1) == total:
            print(f"Classified {idx + 1}/{total} segments")

    return all_records, all_traces


classified_records, classification_traces = classify_all_segments(
    segments=segments,
    model_name=OLLAMA_MODEL
)

print(f"\nFinished initial classification for {len(classified_records)} segments.")
print("\nExample enriched record:")
print(json.dumps(classified_records[0], indent=2, ensure_ascii=False))

Classified 10/21 segments
Classified 20/21 segments
Classified 21/21 segments

Finished initial classification for 21 segments.

Example enriched record:
{
  "id": 0,
  "start": 0.0,
  "end": 4.8,
  "text": "Okay, I'm the doctor.",
  "predicted_role": "doctor",
  "confidence": 1.0,
  "short_reason": "The speaker explicitly states 'I'm the doctor' in the context.",
  "context_text_used": "[TARGET id=0 start=0.00 end=4.80] Okay, I'm the doctor.\n[NEXT id=1 start=4.80 end=5.84] Good morning."
}


In [12]:
# Cell 9: Post-process and smooth labels with context

# This cell is HEURISTIC.
# The model-based step already happened above.
# Here we only apply a small readable smoothing rule:
# If a very short / ambiguous utterance has low confidence and both neighbors agree,
# flip it to the neighbor role.

AMBIGUOUS_SHORT_PHRASES = {
    "okay", "ok", "alright", "all right", "i see", "mm-hmm", "mhm", "uh-huh",
    "right", "sure", "thanks", "thank you", "got it", "sounds good", "don't worry"
}


def is_short_ambiguous_utterance(text: str) -> bool:
    clean = normalize_whitespace(text).lower().strip(" .,!?:;")
    word_count = len(clean.split())

    if clean in AMBIGUOUS_SHORT_PHRASES:
        return True

    if word_count <= 3:
        return True

    return False


def smooth_labels(
    records: List[Dict[str, Any]],
    low_conf_threshold: float = LOW_CONFIDENCE_THRESHOLD,
    enabled: bool = ENABLE_SMOOTHING
) -> List[Dict[str, Any]]:
    """
    Simple readable smoothing pass.
    This does not replace the model.
    It only flips obviously inconsistent short utterances when neighbors agree.
    """
    smoothed = [dict(r) for r in records]

    if not enabled or len(smoothed) < 3:
        return smoothed

    for idx in range(1, len(smoothed) - 1):
        prev_role = smoothed[idx - 1]["predicted_role"]
        curr_role = smoothed[idx]["predicted_role"]
        next_role = smoothed[idx + 1]["predicted_role"]

        curr_text = smoothed[idx]["text"]
        curr_conf = float(smoothed[idx]["confidence"])

        should_consider = (
            is_short_ambiguous_utterance(curr_text)
            and curr_conf < low_conf_threshold
            and prev_role == next_role
            and curr_role != prev_role
        )

        if should_consider:
            smoothed[idx]["predicted_role"] = prev_role
            smoothed[idx]["confidence"] = round(max(curr_conf, 0.55), 2)
            smoothed[idx]["short_reason"] = "Neighbor smoothing for short ambiguous utterance."

    return smoothed


final_records = smooth_labels(classified_records)

num_changed = sum(
    1 for before, after in zip(classified_records, final_records)
    if before["predicted_role"] != after["predicted_role"]
)

print(f"Smoothing enabled: {ENABLE_SMOOTHING}")
print(f"Labels changed by smoothing: {num_changed}")

Smoothing enabled: True
Labels changed by smoothing: 0


In [13]:
# Cell 10: Save output JSON and CSV

def export_results(
    original_json: Dict[str, Any],
    enriched_records: List[Dict[str, Any]],
    input_path: Path,
    output_stem: str
) -> Tuple[Path, Path, pd.DataFrame]:
    """
    Save:
      1. enriched JSON
      2. flat CSV
      3. pandas DataFrame for inspection in the notebook
    """
    output_dir = input_path.parent
    output_json_path = output_dir / f"{output_stem}.json"
    output_csv_path = output_dir / f"{output_stem}.csv"

    export_payload = {
        "source_json_file": str(input_path),
        "audio_path": original_json.get("audio_path"),
        "language": original_json.get("language"),
        "transcript_model": original_json.get("model"),
        "classifier_backend": "ollama",
        "classifier_model": OLLAMA_MODEL,
        "context_back": CONTEXT_BACK,
        "context_forward": CONTEXT_FORWARD,
        "segments": enriched_records,
    }

    output_json_path.write_text(
        json.dumps(export_payload, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    df = pd.DataFrame(enriched_records)
    df.to_csv(output_csv_path, index=False, encoding="utf-8")

    return output_json_path, output_csv_path, df


output_json_path, output_csv_path, results_df = export_results(
    original_json=transcript_data,
    enriched_records=final_records,
    input_path=input_path,
    output_stem=output_stem
)

print(f"Saved JSON to: {output_json_path}")
print(f"Saved CSV  to: {output_csv_path}")

Saved JSON to: transcription_outputs/classified_segments.json
Saved CSV  to: transcription_outputs/classified_segments.csv


In [14]:
# Cell 11: Inspect results

print("Preview of final DataFrame:")
display(results_df.head(10))

print("\nSummary count by predicted role:")
print(results_df["predicted_role"].value_counts(dropna=False))

print("\nAverage confidence by role:")
print(results_df.groupby("predicted_role")["confidence"].mean().round(3))

# Likely review candidates:
# - low confidence
# - or very short utterances that are easy to misclassify
token_counts = results_df["text"].fillna("").apply(lambda x: len(str(x).split()))

review_mask = (
    (results_df["confidence"] < LOW_CONFIDENCE_THRESHOLD) |
    (token_counts <= 3)
)

review_df = results_df.loc[review_mask, [
    "id", "start", "end", "predicted_role", "confidence", "text", "short_reason"
]].copy()

review_df = review_df.sort_values(by=["confidence", "start"], ascending=[True, True])

print("\nLikely misclassifications or low-confidence cases to review:")
display(review_df.head(25))

print("\nExample trace for the first classified segment:")
print(json.dumps(classification_traces[0], indent=2, ensure_ascii=False)[:3000])

Preview of final DataFrame:


,id,start,end,text,predicted_role,confidence,short_reason,context_text_used
0,0,0.00,4.80,"Okay, I'm the doctor.",doctor,1.00,The speaker explicitly states 'I'm the doctor'...,"[TARGET id=0 start=0.00 end=4.80] Okay, I'm th..."
1,1,4.80,5.84,Good morning.,doctor,1.00,The doctor typically initiates the conversatio...,"[PREV id=0 start=0.00 end=4.80] Okay, I'm the ..."
2,2,5.84,7.32,Can you tell me what brings you in today?,doctor,1.00,Asking a question about the patient's condition,[PREV id=1 start=4.80 end=5.84] Good morning.\...
3,3,7.32,11.84,"Uh, yeah, I've been refilling really tired lat...",patient,0.95,Patient describes symptoms of fatigue and exha...,[PREV id=2 start=5.84 end=7.32] Can you tell m...
4,4,11.84,12.84,exhausted.,patient,0.90,The patient is describing a symptom (exhaustio...,"[PREV id=3 start=7.32 end=11.84] Uh, yeah, I'v..."
5,5,12.84,13.84,I see.,doctor,0.95,Doctor's response is typically reassuring and ...,[PREV id=4 start=11.84 end=12.84] exhausted.\n...
6,6,13.84,17.48,Have you noticed any other symptoms like heada...,doctor,0.95,Asking about additional symptoms,[PREV id=5 start=12.84 end=13.84] I see.\n[TAR...
7,7,17.48,20.96,"Sometimes I get this weird chest tightness, bu...",patient,0.95,Patient describes personal symptoms,[PREV id=6 start=13.84 end=17.48] Have you not...
8,8,20.96,21.96,else.,patient,0.95,The response 'else' is a continuation of the p...,[PREV id=7 start=17.48 end=20.96] Sometimes I ...
9,9,21.96,22.96,"Okay, that's important.",doctor,0.95,Doctor typically provides reassurance and asks...,[PREV id=8 start=20.96 end=21.96] else.\n[TARG...



Summary count by predicted role:
predicted_role
doctor     13
patient     8
Name: count, dtype: int64

Average confidence by role:
predicted_role
doctor     0.962
patient    0.938
Name: confidence, dtype: float64

Likely misclassifications or low-confidence cases to review:


,id,start,end,predicted_role,confidence,text,short_reason
4,4,11.84,12.84,patient,0.90,exhausted.,The patient is describing a symptom (exhaustio...
18,18,44.08,45.08,doctor,0.90,Don't worry.,Response to patient's concern with reassurance
5,5,12.84,13.84,doctor,0.95,I see.,Doctor's response is typically reassuring and ...
8,8,20.96,21.96,patient,0.95,else.,The response 'else' is a continuation of the p...
9,9,21.96,22.96,doctor,0.95,"Okay, that's important.",Doctor typically provides reassurance and asks...
15,15,34.84,36.76,patient,0.95,"Alright, cool.","The response 'Alright, cool.' is typically a c..."
16,16,36.76,40.28,doctor,0.95,From what?,Doctors often ask for more information or clar...
1,1,4.80,5.84,doctor,1.00,Good morning.,The doctor typically initiates the conversatio...



Example trace for the first classified segment:
{
  "segment_index": 0,
  "segment_id": 0,
  "prompt": "You are a strict binary classifier for transcript segments from a medical conversation.\n\nTask:\nDetermine whether the TARGET segment was spoken by the doctor or the patient.\n\nUse the TARGET segment plus the nearby context.\nDo NOT rely on speaker diarization labels.\nDo NOT invent facts outside the given text.\n\nGeneral cues:\n- Doctor speech may include questions about symptoms, medication, history, tests, assessment, advice, reassurance, and next steps.\n- Patient speech may include symptoms, lived experience, feelings, habits, timing, history, and answers to intake questions.\n- Very short utterances like \"okay\", \"I see\", \"alright\", \"mm-hmm\", \"don't worry\" must be interpreted using the surrounding context.\n\nReturn STRICT JSON ONLY.\nThe JSON must contain exactly these fields:\n- predicted_role\n- confidence\n- short_reason\n\nRules:\n- predicted_role must be exac